# Third Kaggle run: attacks, and the start-of-text fix

Settings: GPU T4 x2, internet on, secret `GITHUB_TOKEN`, and the outputs of the first two runs added as inputs.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/omarht27/textgrain-eval.git
%cd /kaggle/working/textgrain-eval
!git remote set-url origin https://github.com/omarht27/textgrain-eval.git
!pip install -q -e . --no-deps
sys.path.insert(0, "/kaggle/working/textgrain-eval/src")

os.makedirs("results", exist_ok=True)
NEEDED = ["secret.key", "gumbel.jsonl", "greenlist_d2.jsonl", "textgrain_B4_b0.25.jsonl", "textgrain_B8_b0.25.jsonl",
          "textgrain_B8_b0.25_L1.jsonl", "textgrain_B8_b0.25_L4.jsonl"]
for name in NEEDED:
    shutil.copy(glob.glob(f"/kaggle/input/**/results/{name}", recursive=True)[0], "results/")
!ls results

In [ ]:
MODEL = "--model Qwen/Qwen2.5-1.5B-Instruct --batch-size 16 --method textgrain --beta 0.25 --blocks 8 --columns 8 --min-context 2"
JOBS = {
    0: [
        f"python experiments/generate.py {MODEL} --n-prompts 200 --max-new-tokens 400 --no-eos --out results/textgrain_B8_b0.25_start.jsonl",
        f"python experiments/generate.py {MODEL} --prompts questions --n-prompts 30 --samples 10 --max-new-tokens 150 --out results/questions_textgrain_B8_b0.25_start.jsonl",
        "python experiments/perplexity.py results/textgrain_B8_b0.25_L1.jsonl results/textgrain_B8_b0.25_L4.jsonl results/textgrain_B8_b0.25_start.jsonl",
    ],
    1: [
        "python experiments/attack.py results/gumbel.jsonl results/greenlist_d2.jsonl results/textgrain_B4_b0.25.jsonl results/textgrain_B8_b0.25.jsonl",
    ],
}
processes = []
for gpu, jobs in JOBS.items():
    log = open(f"results/log3_gpu{gpu}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    processes.append(subprocess.Popen("; ".join(jobs), shell=True, env=env, stdout=log, stderr=subprocess.STDOUT))
for p in processes:
    p.wait()
!tail -n 5 results/log3_gpu0.txt results/log3_gpu1.txt
!cd results && zip -q ../run3.zip *start* *.attacks.json *L1.ppl.json *L4.ppl.json log3_*
!ls -la run3.zip